In [1]:
!wget https://raw.githubusercontent.com/cbuntain/umd.inst414/refs/heads/main/Module05/movie_to_cluster.csv

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import json

movie_genre_map = {}
with open("imdb_movies_2000to2022.prolific.json", "r") as in_file:
    for line in in_file:
        this_movie = json.loads(line)
        movie_genre_map[this_movie["imdb_id"]] = this_movie["genres"]

print("Known Movies:", len(movie_genre_map))
movie_genre_map

Known Movies: 20620


{'tt0035423': ['Comedy', 'Fantasy', 'Romance'],
 'tt0088751': ['Comedy', 'Horror', 'Sci-Fi'],
 'tt0096056': ['Drama'],
 'tt0113092': ['Action', 'Adventure', 'Drama'],
 'tt0116391': ['Action', 'Crime', 'Drama'],
 'tt0117461': ['Comedy', 'Romance'],
 'tt0117743': ['Drama', 'Romance'],
 'tt0118652': ['Comedy', 'Horror', 'Mystery'],
 'tt0118852': ['Drama'],
 'tt0118926': ['Crime', 'Drama', 'Thriller'],
 'tt0119004': ['Comedy', 'Drama'],
 'tt0119231': ['Comedy'],
 'tt0119273': ['Action', 'Adventure', 'Animation'],
 'tt0120166': ['Adventure', 'Family', 'Fantasy'],
 'tt0120202': ['Comedy', 'Drama'],
 'tt0120467': ['Crime', 'Drama', 'Thriller'],
 'tt0120630': ['Adventure', 'Animation', 'Comedy'],
 'tt0120667': ['Action', 'Adventure', 'Fantasy'],
 'tt0120679': ['Biography', 'Drama', 'Romance'],
 'tt0120681': ['Horror', 'Mystery', 'Thriller'],
 'tt0120698': ['Comedy', 'Drama', 'Thriller'],
 'tt0120737': ['Action', 'Adventure', 'Drama'],
 'tt0120753': ['Drama', 'Mystery', 'Thriller'],
 'tt0120755

In [3]:
import pandas as pd

df = pd.read_csv('movie_to_cluster.csv')
df

,movie_id,cluster
0,tt0035423,8
1,tt0088751,12
2,tt0096056,6
3,tt0113092,3
4,tt0116391,3
...,...,...
20615,tt9906278,10
20616,tt9906644,13
20617,tt9906844,10
20618,tt9907032,10


In [4]:
# Pr(Cluster)

df.value_counts('cluster') / len(movie_genre_map)

cluster
6     0.154074
0     0.150194
15    0.085063
13    0.082687
2     0.072890
12    0.071096
1     0.066731
3     0.060136
14    0.043307
8     0.037536
10    0.036906
4     0.031765
11    0.031038
7     0.030795
5     0.027158
9     0.018623
Name: count, dtype: float64

In [5]:
# Pr(Genre)

genre_count = {}

for genres in movie_genre_map.values():
    for genre in genres:
        genre_count[genre] = genre_count.get(genre, 0) + 1
genre_count 

{'Comedy': 6017,
 'Fantasy': 877,
 'Romance': 2521,
 'Horror': 3077,
 'Sci-Fi': 1055,
 'Drama': 10157,
 'Action': 3743,
 'Adventure': 1667,
 'Crime': 2773,
 'Mystery': 1535,
 'Thriller': 4004,
 'Animation': 620,
 'Family': 978,
 'Biography': 811,
 'Sport': 441,
 'Music': 390,
 'Western': 211,
 'War': 232,
 'History': 435,
 'Musical': 168,
 'Documentary': 796,
 '': 144,
 'Short': 1,
 'News': 28,
 'Reality-TV': 2}

In [6]:
for target_genre in ["Sci-Fi", "Fantasy", "Adventure"]:
    for cluster_id, group in df.groupby("cluster"):
        this_cluster_genre_count = sum([1 if target_genre in movie_genre_map[movie] else 0 for movie in group["movie_id"]])
    
        # P(Genre | Cluster)
        pr_genre_given_cluster = this_cluster_genre_count / len(group)

    
    # P(Genre, Cluster) = P(Genre | Cluster) = P(Cluster)
    joint_pr_genre_cluster = pr_genre_given_cluster * group.shape[0] / df.shape[0]


    #TARGET: Pr(Cluster | genre "Sci-fi") = Pr(genre, Cluster) / Pr(Genre)
    pr_genre = genre_count[target_genre] / len(movie_genre_map)
    pr_cluster_given_genre = joint_pr_genre_cluster / pr_genre
    print(f"Pr(Cluster {cluster_id} | {target_genre}): {pr_cluster_given_genre}")
print()

Pr(Cluster 15 | Sci-Fi): 0.13175355450236967
Pr(Cluster 15 | Fantasy): 0.020524515393386546
Pr(Cluster 15 | Adventure): 0.04379124175164967



In [8]:

for target_genre in ["Sci-Fi", "Fantasy", "Adventure"]:
    print(f"\nGenre: {target_genre}")

    for cluster_id, group in df.groupby("cluster"):
        this_cluster_genre_count = sum(
            [1 if target_genre in movie_genre_map[movie] else 0
             for movie in group["movie_id"]]
        )

        # P(Genre | Cluster)
        pr_genre_given_cluster = this_cluster_genre_count / len(group)

        # P(Genre, Cluster) = P(Genre | Cluster) * P(Cluster)
        joint_pr_genre_cluster = pr_genre_given_cluster * group.shape[0] / df.shape[0]

        # P(Genre)
        pr_genre = genre_count[target_genre] / len(movie_genre_map)

        # P(Cluster | Genre) = P(Genre, Cluster) / P(Genre)
        pr_cluster_given_genre = joint_pr_genre_cluster / pr_genre

        print(f"Pr[Cluster {cluster_id:02d} | {target_genre}]: \t {pr_cluster_given_genre}")


Genre: Sci-Fi
Pr[Cluster 00 | Sci-Fi]: 	 0.17345971563981044
Pr[Cluster 01 | Sci-Fi]: 	 0.040758293838862564
Pr[Cluster 02 | Sci-Fi]: 	 0.20379146919431282
Pr[Cluster 03 | Sci-Fi]: 	 0.02464454976303318
Pr[Cluster 04 | Sci-Fi]: 	 0.05308056872037915
Pr[Cluster 05 | Sci-Fi]: 	 0.0
Pr[Cluster 06 | Sci-Fi]: 	 0.061611374407582936
Pr[Cluster 07 | Sci-Fi]: 	 0.0
Pr[Cluster 08 | Sci-Fi]: 	 0.008530805687203791
Pr[Cluster 09 | Sci-Fi]: 	 0.021800947867298578
Pr[Cluster 10 | Sci-Fi]: 	 0.0009478672985781991
Pr[Cluster 11 | Sci-Fi]: 	 0.047393364928909956
Pr[Cluster 12 | Sci-Fi]: 	 0.18862559241706164
Pr[Cluster 13 | Sci-Fi]: 	 0.013270142180094787
Pr[Cluster 14 | Sci-Fi]: 	 0.03033175355450237
Pr[Cluster 15 | Sci-Fi]: 	 0.13175355450236967

Genre: Fantasy
Pr[Cluster 00 | Fantasy]: 	 0.23831242873432154
Pr[Cluster 01 | Fantasy]: 	 0.009122006841505131
Pr[Cluster 02 | Fantasy]: 	 0.18814139110604333
Pr[Cluster 03 | Fantasy]: 	 0.0330672748004561
Pr[Cluster 04 | Fantasy]: 	 0.05473204104903078
P